# GameRec: Exploratory Data Analysis (EDA)
**Dataset**: Game Recommendations on Steam (Anton Kozyriev, Kaggle)
**Goal**: Memory-efficient data profiling, schema validation, power-law distribution analysis, and catalog coverage inspection.

In [1]:
import sys
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Setup path to src
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from gamerec.data import load_processed_catalog, load_processed_interactions, get_default_paths
paths = get_default_paths(project_root)
print("Project paths initialized:", paths["project_root"])

## 1. Load Processed Game Catalog
Inspect unified metadata, community tags, pricing, and Bayesian shrinkage ratings.

In [2]:
catalog_df = load_processed_catalog(paths["processed_catalog"])
print(f"Catalog Shape: {catalog_df.shape}")
catalog_df.select(["app_id", "title", "positive_ratio", "user_reviews", "price_final", "bayesian_rating", "has_tags"]).head(5)

## 2. Load Active Benchmark Interactions
Inspect the 10-core active user benchmark dataset created for collaborative filtering and ranking evaluation.

In [3]:
benchmark_df = load_processed_interactions(paths["processed_benchmark"], lazy=False)
print(f"Benchmark Interactions Shape: {benchmark_df.shape}")
print(f"Unique Users: {benchmark_df['user_id'].n_unique():,}")
print(f"Unique Games: {benchmark_df['app_id'].n_unique():,}")
benchmark_df.head(5)

## 3. Explicit Feedback vs Missing Feedback Semantics
- : Explicit Positive Recommendation (Thumbs Up)
- : Explicit Negative Recommendation (Thumbs Down)
- Unobserved interactions: Missing feedback (NEVER imputed as negative)

In [4]:
pos_count = (benchmark_df["is_recommended"] == True).sum()
neg_count = (benchmark_df["is_recommended"] == False).sum()
print(f"Positive interactions: {pos_count:,} ({pos_count/len(benchmark_df)*100:.2f}%)")
print(f"Negative interactions: {neg_count:,} ({neg_count/len(benchmark_df)*100:.2f}%)")

## 4. Run Full EDA Visualizations

In [5]:
from scripts.run_eda import setup_plotting_style, plot_game_popularity, plot_top_tags, plot_user_activity, plot_sentiment_balance, plot_playtime_distribution, plot_missingness_summary
setup_plotting_style()
figures_dir = paths["reports_dir"] / "figures"
plot_game_popularity(catalog_df, figures_dir)
plot_top_tags(catalog_df, figures_dir)
plot_user_activity(benchmark_df, figures_dir)
plot_sentiment_balance(benchmark_df, figures_dir)
plot_playtime_distribution(benchmark_df, figures_dir)
plot_missingness_summary(catalog_df, figures_dir)
print("Visualizations generated successfully!")